# Demand Forecasting for Inventory Planning (M5 Dataset)

This notebook focuses on forecasting daily demand for top-selling SKUs in a single store and evaluating forecast accuracy to support inventory replenishment decisions.


## Business Objective

The objective of this analysis is to forecast daily SKU-level demand and quantify demand uncertainty in order to support inventory planning decisions such as safety stock and reorder point calculation.


## Scope & Assumptions

- Store-level analysis (single store)
- Top 50 SKUs by historical demand
- Daily demand forecasting
- Fixed lead time of 7 days
- Continuous review inventory policy
- Forecast accuracy evaluated using a 28-day holdout window


## Dataset Description

The analysis uses the M5 Forecasting dataset, which contains historical daily unit sales for Walmart stores, along with calendar and pricing information.

Key datasets used:
- sales_train_validation.csv: historical daily unit sales
- calendar.csv: calendar attributes and dates
- sell_prices.csv: item pricing by store and week


## Evaluation Metrics

- sMAPE (primary): used to compare forecast accuracy across SKUs
- MAPE (secondary): used for interpretability
- Mean daily demand and demand variability are computed for inventory planning


## Notebook Roadmap

1. Load and prepare raw data
2. Reduce scope to a single store and top 50 SKUs
3. Convert data to time-series format
4. Implement baseline forecasting model
5. Implement improved forecasting model
6. Evaluate forecast accuracy on a holdout window
7. Save forecast performance outputs for inventory planning


In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/"


In [2]:
sales = pd.read_csv(DATA_PATH + "sales_train_validation.csv")
calendar = pd.read_csv(DATA_PATH + "calendar.csv")
prices = pd.read_csv(DATA_PATH + "sell_prices.csv")

In [3]:
print("sales:", sales.shape)
print("calendar:", calendar.shape)
print("prices:", prices.shape)

print("\nSales columns (first 12):", list(sales.columns[:12]))
print("Last 5 demand columns:", [c for c in sales.columns if c.startswith("d_")][-5:])

sales.head(2)


sales: (30490, 1919)
calendar: (1969, 14)
prices: (6841121, 4)

Sales columns (first 12): ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6']
Last 5 demand columns: ['d_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


## Scope Reduction: Store and SKU Selection

To keep the analysis practical and aligned with store-level planning, the dataset is filtered to a single store (CA_1).  
Within this store, the top 50 SKUs by total historical demand are selected for forecasting and inventory analysis.


In [4]:
STORE_ID = "CA_1"

sales_store = sales[sales["store_id"] == STORE_ID].copy()

sales_store.shape


(3049, 1919)

In [5]:
demand_cols = [c for c in sales_store.columns if c.startswith("d_")]

sales_store["total_demand"] = sales_store[demand_cols].sum(axis=1)

top_50 = (
    sales_store
    .sort_values("total_demand", ascending=False)
    .head(50)
    .copy()
)

top_50.shape

(50, 1920)

In [6]:
top_50 = top_50.drop(columns=["total_demand"])

top_50.head(2)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
2314,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,107,182,47,47,...,47,82,83,30,45,29,53,87,95,42
2810,FOODS_3_586_CA_1_validation,FOODS_3_586,FOODS_3,FOODS,CA_1,CA,42,36,30,23,...,32,37,64,30,40,31,45,28,40,54


## Time Series Transformation

The wide-format demand data is converted into a long time-series format, where each row represents daily demand for a given SKU and date.


In [7]:
demand_long = top_50.melt(
    id_vars=["id", "item_id", "store_id"],
    value_vars=demand_cols,
    var_name="d",
    value_name="demand"
)

demand_long.shape


(95650, 5)

In [8]:
demand_long = demand_long.merge(
    calendar[["d", "date"]],
    on="d",
    how="left"
)

demand_long["date"] = pd.to_datetime(demand_long["date"])

demand_long.head()


,id,item_id,store_id,d,demand,date
0,FOODS_3_090_CA_1_validation,FOODS_3_090,CA_1,d_1,107,2011-01-29
1,FOODS_3_586_CA_1_validation,FOODS_3_586,CA_1,d_1,42,2011-01-29
2,FOODS_3_252_CA_1_validation,FOODS_3_252,CA_1,d_1,19,2011-01-29
3,FOODS_3_120_CA_1_validation,FOODS_3_120,CA_1,d_1,0,2011-01-29
4,FOODS_3_714_CA_1_validation,FOODS_3_714,CA_1,d_1,49,2011-01-29


In [9]:
demand_long = demand_long.sort_values(["id", "date"])

demand_long.groupby("id")["date"].count().head()


id
FOODS_1_085_CA_1_validation    1913
FOODS_1_218_CA_1_validation    1913
FOODS_2_019_CA_1_validation    1913
FOODS_2_197_CA_1_validation    1913
FOODS_2_244_CA_1_validation    1913
Name: date, dtype: int64

## Train–Test Split and Baseline Forecast

A 28-day holdout window is used to evaluate forecast accuracy.  
A seasonal naïve model (weekly seasonality) is implemented as a baseline benchmark.

In [10]:
HOLDOUT_DAYS = 28

def train_test_split(ts):
    return ts.iloc[:-HOLDOUT_DAYS], ts.iloc[-HOLDOUT_DAYS:]

In [11]:
def seasonal_naive_forecast(train_values, horizon, season_length=7):
    last_season = train_values[-season_length:]
    reps = int(np.ceil(horizon / season_length))
    return np.tile(last_season, reps)[:horizon]


In [12]:
baseline_results = []

for series_id, df in demand_long.groupby("id"):
    ts = df.set_index("date")["demand"]
    train, test = train_test_split(ts)

    forecast = seasonal_naive_forecast(
        train.values,
        horizon=len(test),
        season_length=7
    )

    smape = np.mean(
        2 * np.abs(forecast - test.values) /
        (np.abs(forecast) + np.abs(test.values) + 1e-8)
    )

    baseline_results.append({
        "series_id": series_id,
        "model": "seasonal_naive",
        "smape": smape,
        "avg_daily_demand": ts.mean(),
        "std_daily_demand": ts.std(),
        "cv": ts.std() / (ts.mean() + 1e-8)
    })

baseline_results = pd.DataFrame(baseline_results)
baseline_results.head()


,series_id,model,smape,avg_daily_demand,std_daily_demand,cv
0,FOODS_1_085_CA_1_validation,seasonal_naive,0.520839,10.641924,6.947792,0.652870
1,FOODS_1_218_CA_1_validation,seasonal_naive,0.775914,11.702561,9.325353,0.796864
2,FOODS_2_019_CA_1_validation,seasonal_naive,0.508236,18.465238,10.984678,0.594884
3,FOODS_2_197_CA_1_validation,seasonal_naive,0.591558,21.274438,12.736485,0.598676
4,FOODS_2_244_CA_1_validation,seasonal_naive,0.859070,10.476215,6.920014,0.660545


In [13]:
baseline_results.describe()


,smape,avg_daily_demand,std_daily_demand,cv
count,50.000000,50.000000,50.000000,50.000000
mean,0.569184,17.265332,13.985473,0.851080
std,0.305136,10.214669,8.473940,0.310898
min,0.142857,9.881861,4.580489,0.357784
25%,0.401291,11.507710,9.394496,0.595832
50%,0.478181,14.329848,12.827624,0.885515
75%,0.645966,18.381208,15.013522,1.030813
max,1.760334,66.493988,57.862321,1.785420


## Improved Forecasting Model: Exponential Smoothing (ETS)

An Exponential Smoothing model with trend and weekly seasonality is implemented to improve forecast accuracy over the baseline seasonal naïve approach.


In [14]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing


In [17]:
ets_results = []

for series_id, df in demand_long.groupby("id"):
    ts = df.set_index("date")["demand"].asfreq("D")
    train, test = train_test_split(ts)

    try:
        model = ExponentialSmoothing(
            train,
            trend="add",
            seasonal="add",
            seasonal_periods=7
        ).fit(optimized=True)

        forecast = model.forecast(len(test))

        smape = np.mean(
            2 * np.abs(forecast - test.values) /
            (np.abs(forecast) + np.abs(test.values) + 1e-8)
        )

        ets_results.append({
            "series_id": series_id,
            "model": "ETS",
            "smape": smape
        })
    except Exception as e:
        # In case a series fails to converge
        continue

ets_results = pd.DataFrame(ets_results)
ets_results.head()


,series_id,model,smape
0,FOODS_1_085_CA_1_validation,ETS,0.478024
1,FOODS_1_218_CA_1_validation,ETS,0.617933
2,FOODS_2_019_CA_1_validation,ETS,0.400250
3,FOODS_2_197_CA_1_validation,ETS,0.758981
4,FOODS_2_244_CA_1_validation,ETS,0.532942


In [18]:
comparison = baseline_results.merge(
    ets_results,
    on="series_id",
    suffixes=("_baseline", "_ets")
)

comparison["better_model"] = np.where(
    comparison["smape_ets"] < comparison["smape_baseline"],
    "ETS",
    "Seasonal Naive"
)

comparison.head()


,series_id,model_baseline,smape_baseline,avg_daily_demand,std_daily_demand,cv,model_ets,smape_ets,better_model
0,FOODS_1_085_CA_1_validation,seasonal_naive,0.520839,10.641924,6.947792,0.652870,ETS,0.478024,ETS
1,FOODS_1_218_CA_1_validation,seasonal_naive,0.775914,11.702561,9.325353,0.796864,ETS,0.617933,ETS
2,FOODS_2_019_CA_1_validation,seasonal_naive,0.508236,18.465238,10.984678,0.594884,ETS,0.400250,ETS
3,FOODS_2_197_CA_1_validation,seasonal_naive,0.591558,21.274438,12.736485,0.598676,ETS,0.758981,Seasonal Naive
4,FOODS_2_244_CA_1_validation,seasonal_naive,0.859070,10.476215,6.920014,0.660545,ETS,0.532942,ETS


In [19]:
comparison["better_model"].value_counts()

better_model
ETS               33
Seasonal Naive    17
Name: count, dtype: int64

## Save Forecasting Outputs

Forecast accuracy and demand variability metrics are saved for use in inventory planning and dashboarding.


In [20]:
comparison.to_csv(
    "../outputs/forecasts/forecast_accuracy_by_sku.csv",
    index=False
)


In [21]:
pd.read_csv("../outputs/forecasts/forecast_accuracy_by_sku.csv").head()


,series_id,model_baseline,smape_baseline,avg_daily_demand,std_daily_demand,cv,model_ets,smape_ets,better_model
0,FOODS_1_085_CA_1_validation,seasonal_naive,0.520839,10.641924,6.947792,0.652870,ETS,0.478024,ETS
1,FOODS_1_218_CA_1_validation,seasonal_naive,0.775914,11.702561,9.325353,0.796864,ETS,0.617933,ETS
2,FOODS_2_019_CA_1_validation,seasonal_naive,0.508236,18.465238,10.984678,0.594884,ETS,0.400250,ETS
3,FOODS_2_197_CA_1_validation,seasonal_naive,0.591558,21.274438,12.736485,0.598676,ETS,0.758981,Seasonal Naive
4,FOODS_2_244_CA_1_validation,seasonal_naive,0.859070,10.476215,6.920014,0.660545,ETS,0.532942,ETS
